In [3]:
import pandas as pd

eval_df = pd.read_json("../data/processed/eval_v10.jsonl", lines=True)
eval_df.head()

,corrupted,original,verdict,reason
0,Wir geht morgen auf zehn los.,Wir gehen morgen um zehn los.,KEEP,Corrupt sentence is clearly ungrammatical (ver...
1,Wie kann du so ein Mann lieben?,Wie kannst du so einen Mann lieben?,KEEP,Clear grammatical errors (missing verb ending ...
2,Ich kann faulen Menschen nicht aussteht.,Ich kann faule Menschen nicht ausstehen.,KEEP,Clear grammatical errors (incorrect adjective ...
3,Manchmal fühle ich mich wie einen mutterlose K...,Manchmal fühle ich mich wie ein mutterloses Kind.,KEEP,Clear grammatical errors (case and adjective e...
4,Was genau musst er tut?,Was genau muss er tun?,KEEP,Clear subject‑verb agreement and infinitive er...


In [4]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "../models/Ministral-3-3B-Instruct-2512-GEC-v9", # Point directly to the ADAPTER folder
    max_seq_length = 512,          # Use the same as your training
    load_in_4bit = True,            # Set to True if you trained with 4-bit
    dtype = None,                   # Auto-detect (Float16/Bfloat16)
    # device_map="cpu"
)

# Switch model to inference mode (merges LoRA weights for generation)
FastLanguageModel.for_inference(model)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


d:\dev\text-tune-ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0326 20:24:07.858000 228 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Mistral3 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA GeForce RTX 3070 Ti. Num GPUs = 1. Max memory: 8.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 8.6. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 458/458 [00:01<00:00, 342.85it/s, Materializing param=model.vision_tower.transformer.layers.23.ffn_norm.weight]              


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Mistral3ForConditionalGeneration(
      (model): Mistral3Model(
        (vision_tower): PixtralVisionModel(
          (patch_conv): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
          (ln_pre): PixtralRMSNorm((1024,), eps=1e-05)
          (transformer): PixtralTransformer(
            (layers): ModuleList(
              (0-23): 24 x PixtralAttentionLayer(
                (attention_norm): PixtralRMSNorm((1024,), eps=1e-05)
                (feed_forward): PixtralMLP(
                  (gate_proj): lora.Linear(
                    (base_layer): Linear(in_features=1024, out_features=4096, bias=False)
                    (lora_dropout): ModuleDict(
                      (default): Identity()
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=1024, out_features=16, bias=False)
                    )
                    (lora_B): ModuleDict(
 

In [5]:
from src.prompts import get_inference_prompt_v5

eval_df["inference_prompt"] = eval_df["corrupted"].apply(get_inference_prompt_v5)
eval_df.iloc[0]["inference_prompt"]

'<s>[INST]Korrigiere die Grammatik im folgenden Text, aber behalte den ursprünglichen Stil und Ton bei. Verleihe dem Text keine formelle Note, wenn er diese nicht hat. Gib **nur** den korrigierten Satz zurück, ohne Anmerkungen. Wenn der Satz korrekt ist, gib ihn unverändert zurück.\n\nWir geht morgen auf zehn los.[/INST]'

In [6]:
# tokenize the list of prompts
tensors = tokenizer(None, eval_df["inference_prompt"].tolist(), return_tensors="pt", add_special_tokens=False, padding=True).to(model.device)
print(tensors["input_ids"].shape)

torch.Size([354, 102])


In [7]:
from tqdm import tqdm
corrections = []

# Generate corrections for each prompt in batch of 8
for i in tqdm(range(0, len(tensors["input_ids"]), 8), desc="Generating corrections"):
    batch = {key: val[i:i+8] for key, val in tensors.items()}
    output = model.generate(**batch, max_new_tokens=128)

    for j in range(output.shape[0]):
        # Decode only the NEW tokens (skip the input prompt tokens)
        input_length = batch["input_ids"].shape[1]
        generated_tokens = output[j][input_length:]
        output_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
        corrections.append(output_text)

Generating corrections: 100%|██████████| 45/45 [05:26<00:00,  7.26s/it]  


In [8]:
corrections

['Wir gehen morgen um zehn los.',
 'Wie kannst du so einen Mann lieben?',
 'Ich kann faule Menschen nicht ausstehen.',
 'Manchmal fühle ich mich wie ein mutterloses Kind.',
 'Was genau muss er tun?',
 'So kann mich niemand erkennen.',
 'Ich will nicht, dass du mich falsch verstehst.',
 'Geht ihr oft ins Theater?',
 'Im Allgemeinen machen sie im Ausland Urlaub.',
 'Wir müssen unseren Vorfahren ehren.',
 'Ich frühstückte nicht, weil ich meistens zu spät aufstehe und ich keine Zeit mehr dafür habe.',
 'Ich glaube, sie geben zu viel für Kleider aus.',
 'Ich möchte dir nur daran erinnern, was du gestern gesagt hast.',
 'Unser Zug kommt morgen Mittag an.',
 'Ich mag Winter, weil ich viel Ski laufen, Schlitten fahren und Schlittschuh laufen kann.',
 'Glaubst du wirklich, dass ich dich verlassen will?',
 'Warum lässt sich ein altes Ehepaar scheiden?',
 'Jetzt kann ich Ihnen keine Antwort geben. Ich muss es mir nochmal überlegen.',
 'Er ist immer ungeduldig mit mir und er ist eifersüchtig auf m

In [9]:
pd.DataFrame({
    "corrupted": eval_df["corrupted"],
    "model_corrected": corrections,
    "original": eval_df["original"],
}).to_json("../outputs/Text-Tune-Small-v9-on-eval-v10.jsonl", orient="records", lines=True, force_ascii=False)

In [30]:
asd = pd.read_json("../outputs/Text-Tune-Small-v9-on-eval-v9.jsonl-results.jsonl", lines=True)
asd["corrupted"] = eval_df["corrupted"]
# asd.to_json("../outputs/Text-Tune-Small-v9-on-eval-v9.jsonl-results-with-corrupted.jsonl", orient="records", lines=True, force_ascii=False)
asd.head()

,original,model_corrected,is_grammatically_correct,meaning_preserved,reason,corrupted
0,Ich kann faule Menschen nicht ausstehen.,Ich kann faule Menschen nicht ausstehen.,True,True,unchanged and correct,Ich können faule menschen nicht ausstehen.
1,Wie kannst du so einen Mann lieben?,Wie kannst du so einen Mann lieben?,True,True,unchanged and correct,Wie kann du so ein Mann lieben?
2,Wir gehen morgen um zehn los.,Wir gehen morgen um zehn los.,True,True,unchanged and correct,Wir geht morgen von zehn los.
3,So kann mich niemand erkennen.,So kann mich niemand erkennen.,True,True,unchanged and correct,So können mich niemand erkennen.
4,Manchmal fühle ich mich wie ein mutterloses Kind.,Manchmal fühle ich mich wie ein mutterloses Kind.,True,True,unchanged and correct,Manchmal fühle ich mich wie einen mutterlose K...


In [35]:
asd[asd["meaning_preserved"] == False].to_json("../outputs/Text-Tune-Small-v9-on-eval-v9-meaning_not_preserved.jsonl", orient="records", lines=True, force_ascii=False)